# Generality of AI Models — Week 6 (Trustworthy AI)

Runnable illustrations of the lecture's three parts:

1. **Out-of-Distribution (OOD) Generalization** — definition, distribution shifts, why models fail.
2. **Signal Selection & Adversarial Generalization** — spurious vs. causal features, adversarial examples, defenses.
3. **Generalization from Small Sample Sets** — few-shot / metric learning, overfitting on tiny data.

Framework: **PyTorch**. Everything is self-contained and uses small synthetic datasets so each cell runs in seconds on a free Colab CPU.

> Run cells top to bottom. `Runtime → Run all` works too.


In [ ]:
# Setup — PyTorch is preinstalled on Colab.
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

---
## Part 1 — Out-of-Distribution (OOD) Generalization

**In-distribution (ID)** = test data looks like training data. **OOD** = the test distribution *shifts*.
The slides list three shift types:

| Shift | What changes | Intuition |
|---|---|---|
| **Covariate shift** | input distribution `P(x)` | same rule, inputs move (new lighting, new accents) |
| **Label shift** | label distribution `P(y)` | class balance changes at deployment |
| **Concept shift** | the mapping `P(y\|x)` | the rule itself changes |

We build a simple 2-class problem, train on one input region, and test on a shifted region.


In [ ]:
# Two Gaussian blobs (binary classification). We'll shift the TEST inputs.
def make_blobs(n, shift=0.0):
    # class 0 centered near (-1,0), class 1 near (+1,0); 'shift' pushes both along +x
    n0 = n // 2
    x0 = torch.randn(n0, 2) * 0.5 + torch.tensor([-1.0 + shift, 0.0])
    x1 = torch.randn(n - n0, 2) * 0.5 + torch.tensor([ 1.0 + shift, 0.0])
    x = torch.cat([x0, x1]); y = torch.cat([torch.zeros(n0), torch.ones(n - n0)]).long()
    perm = torch.randperm(len(x)); return x[perm], y[perm]

Xtr, ytr = make_blobs(400, shift=0.0)      # training region
Xid, yid = make_blobs(400, shift=0.0)      # in-distribution test
Xood, yood = make_blobs(400, shift=2.5)    # covariate-shifted test (inputs moved)

plt.figure(figsize=(6,4))
plt.scatter(Xtr[:,0], Xtr[:,1], c=ytr, cmap="coolwarm", s=10, label="train")
plt.scatter(Xood[:,0], Xood[:,1], c=yood, cmap="coolwarm", s=10, marker="x", label="OOD test")
plt.legend(); plt.title("Training region vs. covariate-shifted OOD region"); plt.show()

In [ ]:
# A small MLP classifier.
class MLP(nn.Module):
    def __init__(self, d_in=2, d_h=32, d_out=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_h), nn.ReLU(),
                                 nn.Linear(d_h, d_h), nn.ReLU(),
                                 nn.Linear(d_h, d_out))
    def forward(self, x): return self.net(x)

def train(model, X, y, epochs=300, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad(); loss = F.cross_entropy(model(X), y); loss.backward(); opt.step()
    return model

@torch.no_grad()
def acc(model, X, y): return (model(X).argmax(1) == y).float().mean().item()

model = train(MLP(), Xtr, ytr)
print(f"ID  accuracy : {acc(model, Xid, yid):.3f}")
print(f"OOD accuracy : {acc(model, Xood, yood):.3f}   <-- drop under covariate shift")

**Takeaway.** The model is near-perfect in-distribution but degrades on the shifted region.
This is the core message of the slides: *even minor shifts can cause large performance drops*, because
deep nets specialize to the training distribution rather than learning an abstract, shift-invariant rule.

Try changing `shift=2.5` to a larger value and re-run — accuracy falls further.


---
## Part 2 — Signal Selection & Adversarial Generalization

### 2a. Spurious correlations
The slides' running example: a model predicts *"cow"* only when it sees *green pastures* — it latched onto a
**spurious** background signal instead of the **causal** object shape. We reproduce this with a toy dataset where a
spurious feature is *perfectly* predictive at training time but *flips* at test time.


In [ ]:
# Feature 0 = causal signal (weakly aligned with label, stable across environments)
# Feature 1 = spurious signal (strongly aligned with label in TRAIN, flipped in TEST)
def make_env(n, spurious_corr):
    y = torch.randint(0, 2, (n,)).float()
    causal   = (y * 2 - 1) * 1.0 + torch.randn(n) * 1.0          # noisy but stable
    # spurious feature agrees with y with prob = spurious_corr
    agree = (torch.rand(n) < spurious_corr).float()
    spur_label = torch.where(agree.bool(), y, 1 - y)
    spurious = (spur_label * 2 - 1) * 1.0 + torch.randn(n) * 0.3  # low-noise shortcut
    X = torch.stack([causal, spurious], 1); return X, y.long()

Xtr2, ytr2 = make_env(2000, spurious_corr=0.95)   # shortcut works great in training
Xte2, yte2 = make_env(2000, spurious_corr=0.05)   # shortcut REVERSES at deployment

m2 = train(MLP(d_in=2), Xtr2, ytr2, epochs=400)
print(f"Train acc (shortcut available): {acc(m2, Xtr2, ytr2):.3f}")
print(f"Test  acc (shortcut reversed) : {acc(m2, Xte2, yte2):.3f}   <-- collapses")

# Inspect which feature the model leans on via input-gradient sensitivity.
Xte2.requires_grad_(True)
logit = m2(Xte2)[range(len(yte2)), yte2].sum(); logit.backward()
sens = Xte2.grad.abs().mean(0)
print(f"Mean |d logit / d feature|  causal={sens[0]:.3f}  spurious={sens[1]:.3f}")

The model rides the low-noise **spurious** feature, so it wins in training and fails when that feature's
correlation reverses. **Signal selection** = getting the model to rely on the *causal* feature instead.
Below is a minimal flavor of **Invariant Risk Minimization (IRM)**: penalize predictors whose optimality
*differs across environments*, pushing the model toward features that are stable everywhere.


In [ ]:
# Minimal IRMv1: two training environments with DIFFERENT spurious strengths.
# A truly invariant predictor is simultaneously optimal in both -> gradient penalty ~ 0.
envs = [make_env(2000, 0.95), make_env(2000, 0.80)]

def irm_penalty(logits, y, scale):
    loss = F.cross_entropy(logits * scale, y)
    g = torch.autograd.grad(loss, [scale], create_graph=True)[0]
    return (g ** 2).sum()

def train_irm(lmbda=1e4, epochs=500, lr=1e-2):
    model = MLP(d_in=2); opt = torch.optim.Adam(model.parameters(), lr=lr)
    scale = torch.tensor(1.0, requires_grad=True)
    for _ in range(epochs):
        opt.zero_grad(); err = 0.0; pen = 0.0
        for Xe, ye in envs:
            log = model(Xe) * scale
            err = err + F.cross_entropy(log, ye)
            pen = pen + irm_penalty(model(Xe), ye, scale)
        (err + lmbda * pen).backward(); opt.step()
    return model

m_irm = train_irm()
print(f"ERM  test acc (shortcut reversed): {acc(m2,    Xte2, yte2):.3f}")
print(f"IRM  test acc (shortcut reversed): {acc(m_irm, Xte2, yte2):.3f}   <-- more robust")

IRM recovers a chunk of OOD accuracy by refusing to depend on the environment-specific shortcut.
(Exact numbers vary with the seed; the *direction* — IRM ≥ ERM out of distribution — is the point.)

### 2b. Adversarial examples
The slides' panda→gibbon example: a **tiny, human-imperceptible** perturbation flips the prediction.
We craft one with **FGSM** (Fast Gradient Sign Method) on a freshly trained classifier.


In [ ]:
# Train a clean classifier on the stable blobs, then attack it.
clf = train(MLP(), Xtr, ytr)

def fgsm(model, x, y, eps):
    x = x.clone().detach().requires_grad_(True)
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    return (x + eps * x.grad.sign()).detach()   # step in the loss-increasing direction

clean_acc = acc(clf, Xid, yid)
for eps in [0.0, 0.1, 0.3, 0.5]:
    Xadv = fgsm(clf, Xid, yid, eps)
    print(f"eps={eps:<4}  accuracy on adversarial inputs = {acc(clf, Xadv, yid):.3f}")
print(f"\n(clean accuracy was {clean_acc:.3f}) — small eps already hurts.")

### 2c. Defense — adversarial training
The slides list *adversarial training* (train on perturbed examples) as a defense. We retrain, injecting
FGSM examples each step, and compare robustness. This mirrors the slides' noted **trade-off**: robustness
often costs a little clean accuracy and a lot of compute.


In [ ]:
def train_adv(eps=0.3, epochs=400, lr=1e-2):
    model = MLP(); opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        Xadv = fgsm(model, Xtr, ytr, eps)          # attack current model
        loss = F.cross_entropy(model(Xadv), ytr)   # train on the attacked inputs
        loss.backward(); opt.step()
    return model

robust = train_adv(eps=0.3)
print("               clean    adv(eps=0.3)")
for name, m in [("standard", clf), ("adv-trained", robust)]:
    Xadv = fgsm(m, Xid, yid, 0.3)
    print(f"{name:<12} {acc(m, Xid, yid):.3f}    {acc(m, Xadv, yid):.3f}")

---
## Part 3 — Generalization from Small Sample Sets (Few-Shot)

The slides frame few-shot learning as mimicking **human learning efficiency**: generalize from a *handful* of
examples. Two illustrations:

1. **Overfitting on tiny data** — with few samples, a flexible model memorizes and generalizes poorly.
2. **Metric learning / nearest-centroid** — a slides-listed approach: compare embeddings instead of retraining.


In [ ]:
# 1) Overfitting shrinks as sample size grows. Fit a small net on N points, test on many.
def few_shot_curve():
    Xbig, ybig = make_blobs(4000, shift=0.0)  # large held-out test set
    sizes = [4, 8, 16, 32, 64, 128, 256]
    tr_acc, te_acc = [], []
    for n in sizes:
        Xs, ys = make_blobs(n, shift=0.0)
        m = train(MLP(), Xs, ys, epochs=400)
        tr_acc.append(acc(m, Xs, ys)); te_acc.append(acc(m, Xbig, ybig))
    return sizes, tr_acc, te_acc

sizes, tr_acc, te_acc = few_shot_curve()
plt.figure(figsize=(6,4))
plt.plot(sizes, tr_acc, "o-", label="train acc")
plt.plot(sizes, te_acc, "s-", label="test acc")
plt.xscale("log", base=2); plt.xlabel("# training samples"); plt.ylabel("accuracy")
plt.title("Small samples -> big train/test gap (overfitting)"); plt.legend(); plt.show()
for n, a, b in zip(sizes, tr_acc, te_acc): print(f"n={n:<4} train={a:.3f}  test={b:.3f}")

With very few samples the train–test gap is wide (memorization); it closes as data grows. This is the slides'
*"models may still overfit to the few available examples"* and *"spurious features become more influential in
small datasets."*

### Metric learning: a training-free few-shot classifier
Represent each class by the **centroid** (prototype) of its few support examples, then classify query points by
nearest centroid in embedding space. This is the idea behind **Prototypical Networks** and the slides'
*"comparing similarities in embedding spaces."* Here we use raw features as the embedding for clarity.


In [ ]:
def nearest_centroid_fewshot(k_shot=5, n_query=500):
    # k_shot support examples per class; build prototypes; classify queries by distance.
    Xs, ys = make_blobs(2 * k_shot, shift=0.0)
    protos = torch.stack([Xs[ys == c].mean(0) for c in [0, 1]])   # (2, 2)
    Xq, yq = make_blobs(n_query, shift=0.0)
    d = torch.cdist(Xq, protos)                                   # (n_query, 2)
    pred = d.argmin(1)
    return (pred == yq).float().mean().item()

for k in [1, 5, 20]:
    print(f"{k}-shot nearest-centroid accuracy: {nearest_centroid_fewshot(k):.3f}")

Even **1-shot** works reasonably because the two classes are separable — no gradient training needed at test
time. Metric/meta-learning methods generalize this: learn an *embedding* where nearest-prototype classification
transfers to brand-new classes from only a few examples.


---
## Conclusion (mirrors the closing slides)

- **OOD generalization** exposes how brittle models are under distribution shift; ID accuracy overstates real-world reliability.
- **Signal selection** matters because models prefer *spurious* shortcuts; methods like IRM push toward *causal*, transferable signals.
- **Adversarial generalization** is the worst-case cousin of OOD — imperceptible perturbations flip predictions; adversarial training helps at a cost.
- **Few-shot generalization** targets human-like efficiency; tiny datasets amplify overfitting, while metric/meta-learning enable learning from a handful of examples.

Addressing all three is what the lecture calls the path to **robust, adaptable, and trustworthy AI**.

### Things to try
- Increase the OOD `shift`, or make Part 2's spurious feature even lower-noise, and watch failures worsen.
- Sweep FGSM `eps` and the adversarial-training `eps` to trace the robustness/clean-accuracy trade-off.
- Swap the raw-feature embedding in the metric-learning cell for a learned encoder.
